In [55]:
import pyomo.environ as pyo
import pandas as pd
import math
import numpy as np
from collections import defaultdict
from datetime import timedelta
from pyomo.util.infeasible import log_infeasible_constraints
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
from pyomo.opt import TerminationCondition
warnings.simplefilter(action='ignore', category=FutureWarning)

In [56]:
class charging_point():
    def __init__(self, name, ev_id, session_id, ev_capacity, ev_max_power, ev_arrival_soc, ev_arrival, ev_departure, ev_desired_soc):
        self.efficiency = 0.93
        # Input validation using assertions
        assert isinstance(ev_capacity, list), "ev_capacity must be a list"
        assert isinstance(ev_max_power, list), "ev_max_power must be a list"
        assert isinstance(ev_arrival_soc, list), "ev_arrival_soc must be a list"
        assert isinstance(ev_arrival, list), "ev_arrival must be a list"
        assert isinstance(ev_departure, list), "ev_departure must be a list"
        assert isinstance(ev_desired_soc, list), "ev_desired_soc must be a list"

        assert len(ev_arrival) == len(ev_departure), "ev_arrival and ev_departure must have the same length"
        assert len(ev_arrival) == len(ev_desired_soc), "ev_arrival and ev_desired_soc must have the same length"
        assert len(ev_arrival_soc) == len(ev_desired_soc), "ev_arrival_soc and ev_desired_soc must have the same length"
        assert len(ev_capacity) == len(ev_desired_soc), "ev_capacity and ev_desired_soc must have the same length"
        assert len(ev_max_power) == len(ev_desired_soc), "ev_max_power and ev_desired_soc must have the same length"

        for i in range(len(ev_arrival)):
            assert ev_arrival[i] < ev_departure[i], f"ev_arrival[{i}] must be less than ev_departure[{i}]"
            assert 0.2 <= ev_arrival_soc[i] <= 1, f"ev_arrival_soc[{i}] must be between 0.2 and 1"
            assert 0 <= ev_desired_soc[i] <= 1, f"ev_desired_soc[{i}] must be between 0 and 1"
            min_time_to_charge = ev_departure[i] - ev_arrival[i]
            min_req_charge = (ev_desired_soc[i] - ev_arrival_soc[i]) * ev_capacity[i] / self.efficiency
            min_req_charge_per_time = min_req_charge / min_time_to_charge
            assert min_req_charge_per_time <= ev_max_power[i], f"min_req_charge_per_time ({min_req_charge_per_time:.2f}) must be less than or equal to ev_max_power[{i}] ({ev_max_power[i]}) for {name}"
            if min_req_charge_per_time * 4 >= ev_max_power[i]:
                warnings.warn(f"Charging would not work for 15-min resolution for {name}")


        self.name = name
        self.ev_capacity = ev_capacity
        self.ev_max_power = ev_max_power
        self.ev_arrival_soc = ev_arrival_soc
        self.ev_desired_soc = ev_desired_soc
        self.ev_arrival = ev_arrival
        self.ev_departure = ev_departure
        self.num_evs = len(ev_capacity) # Store the number of EVs
        self.ev_id = ev_id
        self.session_id = session_id

class building():
    def __init__(self, name, load, pv_production, bess_capacity, bess_max_power, bess_initial_soc):
        self.efficiency = 0.93
        self.name = name
        self.load = load
        self.pv_production = pv_production
        self.bess_capacity = bess_capacity
        self.bess_max_power = bess_max_power
        self.bess_initial_soc = bess_initial_soc

class LEC_Opt_spot():
    def __init__(self, charging_points, buildings, spot_prices, previous_monthly_peak=0, v2g_on=1, resolution = 1, dc = False, subscription_fee = 605/30, transmission_fee = 0.113,
                 transmission_health_incentive = 0.04, peak_effect_fee = 61.55/30, energy_tax = 0.439, energy_certificate = 0.005, incentive_per_kwh=0.1):
        self.M = 10000
        self.charging_points = charging_points
        self.buildings = buildings
        self.spot_prices = spot_prices
        self.incentive_per_kwh = incentive_per_kwh
        self.v2g_on = v2g_on
        self.previous_monthly_peak = previous_monthly_peak
        self.resolution = resolution
        self.dc = dc
        self.subscription_fee = subscription_fee
        self.tranmission_fee = transmission_fee
        self.tranmission_health_incentive = transmission_health_incentive
        self.peak_effect_fee = peak_effect_fee
        self.energy_tax = energy_tax
        self.energy_certificate = energy_certificate
        self.model = pyo.ConcreteModel()
        self.build_model()

    def build_model(self):
        self.model.T = pyo.Set(initialize=range(len(self.spot_prices)))
        self.model.spot_prices = self.spot_prices
        self.model.previous_monthly_peak = self.previous_monthly_peak
        self.model.monthly_peak = pyo.Var(initialize = 0)
        self.model.P_im_grid = pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 100000))
        self.model.P_ex_grid = pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 100000))
        self.model.B_im_grid = pyo.Var(self.model.T, within=pyo.Binary)
        self.model.Peakload = pyo.Var(within=pyo.NonNegativeReals)
        self.model.transmission_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.supplier_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.overall_dso_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.tax_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.peak_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.Subscription_fee = self.subscription_fee  # Subscription fee SEK/14 days
        self.model.Transmission_fee = self.tranmission_fee  # Electricity transmission fee SEK/kWh
        self.model.Transmission_health_incentive = self.tranmission_health_incentive #Transmission health incentive SEK/kWh
        self.model.Effect_fee = self.peak_effect_fee       # Effect fee SEK/kW/14 days
        self.model.Energy_tax = self.energy_tax           # Tax fee SEK/kWh
        self.model.Energy_certificate = self.energy_certificate   #Energy certificate SEK/kWh
        self.model.compensation_fee = self.incentive_per_kwh    # Transfer compensation fee SEK/kWh
        self.model.resolution = self.resolution

        for charge_point in self.charging_points:
            setattr(self.model, f'{charge_point.name}_P', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000)))

            # Iterate through each EV at the charging point
            for ev_index in range(len(charge_point.session_id)):
                ev_name = f'{charge_point.name}_S{charge_point.session_id[ev_index]}'  # Unique name for each EV
                setattr(self.model, f'{ev_name}_ch', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, charge_point.ev_max_power[ev_index])))
                setattr(self.model, f'{ev_name}_ds', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, self.v2g_on * charge_point.ev_max_power[ev_index])))
                setattr(self.model, f'{ev_name}_soc', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 1)))
                setattr(self.model, f'{ev_name}_Bch', pyo.Var(self.model.T, within=pyo.Binary))

                ev_capacity = charge_point.ev_capacity[ev_index]
                ev_arrival_soc = charge_point.ev_arrival_soc[ev_index]
                ev_arrival = charge_point.ev_arrival[ev_index]
                ev_departure = charge_point.ev_departure[ev_index]
                ev_desired_soc = charge_point.ev_desired_soc[ev_index]

                def ev_soc_rule(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    if t == ev_arrival:
                        return ev_soc == ev_arrival_soc + (ev_ch * charge_point.efficiency - ev_ds / charge_point.efficiency) / ev_capacity / self.resolution
                    elif ev_arrival < t <= ev_departure:
                        ev_previous_soc = getattr(model, f'{ev_name}_soc')[t - 1]
                        return ev_soc == ev_previous_soc + (ev_ch * charge_point.efficiency - ev_ds / charge_point.efficiency) / ev_capacity / self.resolution
                    else:
                        return pyo.Constraint.Skip
                setattr(self.model, f'{ev_name}_soc_constraint', pyo.Constraint(self.model.T, rule=ev_soc_rule))

                def ev_soc_min_rule(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    ev_arrival = charge_point.ev_arrival[ev_index]
                    ev_departure = charge_point.ev_departure[ev_index]
                    if ev_arrival <= t <= ev_departure:
                        return ev_soc >= 0.2
                    return ev_soc == 0
                setattr(self.model, f'{ev_name}_soc_min_constraint', pyo.Constraint(self.model.T, rule=ev_soc_min_rule))

                def ev_max_ch(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_Bch = getattr(model, f'{ev_name}_Bch')[t]
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    return ev_ch <= ev_Bch * charge_point.ev_max_power[ev_index]
                setattr(self.model, f'{ev_name}_max_ch_constraint', pyo.Constraint(self.model.T, rule=ev_max_ch))

                def ev_max_ds(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_Bch = getattr(model, f'{ev_name}_Bch')[t]
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    return ev_ds <= (1 - ev_Bch) * charge_point.ev_max_power[ev_index]
                setattr(self.model, f'{ev_name}_max_ds_constraint', pyo.Constraint(self.model.T, rule=ev_max_ds))

                def ev_avail_ch(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    if charge_point.ev_arrival[ev_index] <= t < charge_point.ev_departure[ev_index]:
                        return ev_ch >= 0
                    return ev_ch == 0
                setattr(self.model, f'{ev_name}_avail_ch_constraint', pyo.Constraint(self.model.T, rule=ev_avail_ch))

                def ev_avail_ds(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    if charge_point.ev_arrival[ev_index] <= t < charge_point.ev_departure[ev_index]:
                        return ev_ds >= 0
                    return ev_ds == 0
                setattr(self.model, f'{ev_name}_avail_ds_constraint', pyo.Constraint(self.model.T, rule=ev_avail_ds))

                def ev_desired_soc(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    if t == charge_point.ev_departure[ev_index]:
                        return ev_soc >= charge_point.ev_desired_soc[ev_index]
                    return pyo.Constraint.Skip
                setattr(self.model, f'{ev_name}_desired_soc_constraint', pyo.Constraint(self.model.T, rule=ev_desired_soc))

            def consumption(model, t, charge_point=charge_point):
                P = getattr(model, f'{charge_point.name}_P')[t]
                ev_power = sum(getattr(model, f'{charge_point.name}_S{charge_point.session_id[ev_index]}_ch')[t] - getattr(model, f'{charge_point.name}_S{charge_point.session_id[ev_index]}_ds')[t] for ev_index in range(charge_point.num_evs))
                return  ev_power == P
            setattr(self.model, f'{charge_point.name}_consumption_constraint', pyo.Constraint(self.model.T, rule=consumption))
        
        #Building constraints:
        for building in self.buildings:
            setattr(self.model, f'{building.name}_P', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000)))
            setattr(self.model, f'{building.name}_bess_soc', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, 1)))
            setattr(self.model, f'{building.name}_bess_ch', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, building.bess_max_power)))
            setattr(self.model, f'{building.name}_bess_ds', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, building.bess_max_power)))
            setattr(self.model, f'{building.name}_bess_Bch', pyo.Var(self.model.T, within=pyo.Binary))

            def bess_soc_rule(model, t, building = building):
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                if building.bess_capacity == 0:
                    return bess_soc == 0
                elif t == 0:
                    return bess_soc == building.bess_initial_soc + (bess_ch*building.efficiency - bess_ds/building.efficiency)/building.bess_capacity
                else:
                    bess_previous_soc = getattr(model, f'{building.name}_bess_soc')[t-1]
                    return bess_soc == bess_previous_soc + (bess_ch*building.efficiency - bess_ds/building.efficiency)/building.bess_capacity
            setattr(self.model, f'{building.name}_bess_soc_constraint', pyo.Constraint(self.model.T, rule = bess_soc_rule))

            def bess_soc_min_rule(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                if building.bess_capacity == 0:
                    return bess_soc == 0
                return bess_soc >= 0.2
            setattr(self.model, f'{building.name}_bess_soc_min_constraint', pyo.Constraint(self.model.T, rule = bess_soc_min_rule))  

            def bess_max_ch(model, t, building = building):
                bess_Bch = getattr(model, f'{building.name}_bess_Bch')[t]
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                if building.bess_capacity == 0:
                    return bess_ch == 0
                return bess_ch <= bess_Bch*building.bess_max_power
            setattr(self.model, f'{building.name}_bess_max_ch_constraint', pyo.Constraint(self.model.T, rule = bess_max_ch))

            def bess_max_ds(model, t, building = building):
                bess_Bch = getattr(model, f'{building.name}_bess_Bch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                if building.bess_capacity == 0:
                    return bess_ds == 0
                return bess_ds <= (1-bess_Bch)*building.bess_max_power
            setattr(self.model, f'{building.name}_bess_max_ds_constraint', pyo.Constraint(self.model.T, rule = bess_max_ds))

            def building_consumption(model, t, building = building):
                P = getattr(model, f'{building.name}_P')[t]
                load = building.load[t]
                pv = building.pv_production[t]
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                return load - pv - bess_ds + bess_ch == P
            setattr(self.model, f'{building.name}_consumption_constraint', pyo.Constraint(self.model.T, rule = building_consumption))

        def power_balance(model, t):
            overall_consumption = sum(getattr(model, f'{charge_point.name}_P')[t] for charge_point in self.charging_points) + \
                                    sum(getattr(model, f'{building.name}_P')[t] for building in self.buildings)
            P_im = self.model.P_im_grid[t]
            P_ex = self.model.P_ex_grid[t]
            return P_im - P_ex == overall_consumption
        self.model.power_balance_constarint = pyo.Constraint(self.model.T, rule=power_balance)

        def power_import(model, t):
            P_im = self.model.P_im_grid[t]
            B_im = self.model.B_im_grid[t]
            return P_im <= self.M * B_im
        self.model.power_import_constraint = pyo.Constraint(self.model.T, rule=power_import)

        def power_export(model, t):
            P_ex = self.model.P_ex_grid[t]
            B_im = self.model.B_im_grid[t]
            return P_ex <= self.M * (1 - B_im)
        self.model.power_export_constraint = pyo.Constraint(self.model.T, rule=power_export)

        def peak_load_constraint(model, t):
            return model.Peakload >= model.P_im_grid[t] - model.P_ex_grid[t]
        self.model.peak_load_constraint = pyo.Constraint(self.model.T, rule=peak_load_constraint)

        def previous_peak_check1(model):
            return model.monthly_peak >= model.Peakload
        self.model.previous_peak_check1_constraint = pyo.Constraint(rule=previous_peak_check1)

        def previous_peak_check2(model):
            return model.monthly_peak >= model.previous_monthly_peak
        self.model.previous_peak_check2_constraint = pyo.Constraint(rule=previous_peak_check2)

        def tranmission_cost(model, t):
            return model.transmission_cost[t] == (model.P_im_grid[t] * model.Transmission_fee - model.P_ex_grid[t] * model.Transmission_health_incentive) / self.resolution
        self.model.tranmission_cost_constraint = pyo.Constraint(self.model.T, rule = tranmission_cost)

        def supplier_cost(model, t):
            return model.supplier_cost[t] == (model.P_im_grid[t] * (model.spot_prices[t] + model.Energy_certificate) - model.P_ex_grid[t] * (model.spot_prices[t] + model.Energy_certificate + model.compensation_fee)) / self.resolution
        self.model.supplier_cost_constraint = pyo.Constraint(self.model.T, rule = supplier_cost)

        def objective_rule(model):
            subscription_fee = model.Subscription_fee
            supplier_cost = sum(model.supplier_cost[t] for t in model.T)
            transmission_cost = sum(model.transmission_cost[t] for t in model.T)
            peak_cost = model.Effect_fee * model.monthly_peak * 5 #Check with "day/month" instead of 5
            dso_cost = transmission_cost + peak_cost + subscription_fee
            tax_cost = (supplier_cost + dso_cost)*0.25 + (1.25 * model.Energy_tax * sum(model.P_im_grid[t] - model.P_ex_grid[t] for t in model.T) / 4)
            overall_cost = dso_cost + tax_cost + supplier_cost
            return overall_cost
        self.model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

        def objective_rule_dc(model):
            ev_soc = sum(t * (getattr(model, f'{charge_point.name}_S{charge_point.session_id[ev_index]}_soc')[t]) for charge_point in self.charging_points for ev_index in range(charge_point.num_evs) for t in model.T)
            subscription_fee = model.Subscription_fee
            supplier_cost = sum(model.supplier_cost[t] for t in model.T)
            transmission_cost = sum(model.transmission_cost[t] for t in model.T)
            peak_cost = model.Effect_fee * model.monthly_peak * 5 #Check with "day/month" instead of 5
            dso_cost = transmission_cost + peak_cost + subscription_fee
            tax_cost = (supplier_cost + dso_cost)*0.25 + (1.25 * model.Energy_tax * sum(model.P_im_grid[t] - model.P_ex_grid[t] for t in model.T) / 4)
            overall_cost = dso_cost + tax_cost + supplier_cost - ev_soc * 1000
            return overall_cost
        self.model.obj_dc = pyo.Objective(rule=objective_rule_dc, sense = pyo.minimize)

    def solve(self):
        solver = pyo.SolverFactory('gurobi')
        if self.dc:
            self.model.obj.deactivate()
            self.model.obj_dc.activate()
        else:
            self.model.obj.activate()
            self.model.obj_dc.deactivate()
        self.results = solver.solve(self.model)
        return self.results

    def get_results(self):
        print(f'Objective value: {pyo.value(self.model.obj)}')
        print(f"⏱ Time steps in model: {len(self.model.T)}")
        results = {}
        for cp in self.charging_points:
            T = list(self.model.T)
            ch = []
            ds = []
            soc = []
            ev_connection = []
            session_connection = []

            for t in T:
                ch_t = 0.0
                ds_t = 0.0
                connected_evs_t = None
                soc_ev_t = 0.0
                session_id_t = None

                for ev in range(cp.num_evs):
                    ev_name = f"{cp.name}_S{cp.session_id[ev]}"

                    ch_t += pyo.value(getattr(self.model, f"{ev_name}_ch")[t])
                    ds_t += pyo.value(getattr(self.model, f"{ev_name}_ds")[t])

                    soc_ev = pyo.value(getattr(self.model, f"{ev_name}_soc")[t])
                    soc_ev_t += soc_ev

                    if cp.ev_arrival[ev] <= t < cp.ev_departure[ev]:
                        connected_evs_t = cp.ev_id[ev]
                        session_id_t = cp.session_id[ev]
                
                ev_connection.append(connected_evs_t)
                session_connection.append(session_id_t)

                ch.append(ch_t)
                ds.append(ds_t)
                soc.append(soc_ev_t)

            # ---------- STORE CP-LEVEL RESULTS ----------
            results[f"{cp.name}_EV_connection"] = ev_connection
            results[f"{cp.name}_Session"] = session_connection
            results[f"{cp.name}_ch"] = ch
            results[f"{cp.name}_ds"] = ds
            results[f"{cp.name}_soc"] = soc

            # ---------- EXISTING CP VARIABLES ----------
            results[f"{cp.name}_P"] = [pyo.value(getattr(self.model, f"{cp.name}_P")[t]) for t in T]

        for building in self.buildings:
            results[f'{building.name}_bess_ch'] = [pyo.value(getattr(self.model, f'{building.name}_bess_ch')[t]) for t in self.model.T]
            results[f'{building.name}_bess_ds'] = [pyo.value(getattr(self.model, f'{building.name}_bess_ds')[t]) for t in self.model.T]
            results[f'{building.name}_bess_soc'] = [pyo.value(getattr(self.model, f'{building.name}_bess_soc')[t]) for t in self.model.T]
            results[f'{building.name}_P'] = [pyo.value(getattr(self.model, f'{building.name}_P')[t]) for t in self.model.T]
        results['P_import'] = [pyo.value(self.model.P_im_grid[t]) for t in self.model.T]
        results['P_export'] = [pyo.value(self.model.P_ex_grid[t]) for t in self.model.T]
        results['Transmission cost'] = [pyo.value(self.model.transmission_cost[t]) for t in self.model.T]
        results['Supplier cost'] = [pyo.value(self.model.supplier_cost[t]) for t in self.model.T]
        return pd.DataFrame(results)

In [57]:
#name, ev_capacity, ev_max_power, ev_arrival_soc, ev_arrival, ev_departure, ev_desired_soc
T = 24

# Spot prices (€/kWh)
spot_prices = [0.05 + 0.01*np.sin(i*np.pi/12) for i in range(T)]

# 24-hour realistic load and PV
load = [15 + 0.5*np.sin(i*np.pi/12) for i in range(T)]
pv_production = [0.0 if i < 6 or i > 18 else 1.5*np.sin((i-6)*np.pi/12) for i in range(T)]
load1 = [10 + 0.5*np.sin(i*np.pi/12) for i in range(T)]
pv_production1 = [0.0 if i < 6 or i > 18 else 2.5*np.sin((i-6)*np.pi/12) for i in range(T)]

cp1 = charging_point(name = 'cp1', ev_id = ['sq', 1], session_id= [1, 2], ev_capacity=[45, 65], ev_max_power=[10, 12], ev_arrival=[6, 18], ev_departure= [12, 23], ev_arrival_soc=[0.5, 0.3], ev_desired_soc=[0.75, 0.4])
cp2 = charging_point(name = 'cp2', ev_id = [3, 2], session_id= [1, 2], ev_capacity=[55, 95], ev_max_power=[10, 12], ev_arrival=[8, 15], ev_departure= [12, 20], ev_arrival_soc=[0.6, 0.7], ev_desired_soc=[0.75, 0.75])

b1 = building(name= 'b1', load = load, pv_production=pv_production, bess_capacity=100, bess_initial_soc=0.5, bess_max_power=15)
b2 = building(name= 'b2', load = load1, pv_production=pv_production1, bess_capacity=80, bess_initial_soc=0.5, bess_max_power=7.5)
b3 = building(name= 'b3', load = [0*i for i in range(len(load1))], pv_production=[0*i for i in range(len(pv_production1))], bess_capacity=80, bess_initial_soc=0.5, bess_max_power=7.5)

opt_model = LEC_Opt_spot([cp1, cp2], [b1, b2, b3], spot_prices, dc = True)
results_df = opt_model.solve()
df = opt_model.get_results()
df

Objective value: 580.0060923091596
⏱ Time steps in model: 24


,cp1_EV_connection,cp1_Session,cp1_ch,cp1_ds,cp1_soc,cp1_P,cp2_EV_connection,cp2_Session,cp2_ch,cp2_ds,...,b2_bess_soc,b2_P,b3_bess_ch,b3_bess_ds,b3_bess_soc,b3_P,P_import,P_export,Transmission cost,Supplier cost
0,None,NaN,0.000000,0.0,0.000000,-0.000000,NaN,NaN,0.000000,0.0,...,0.500000,10.000000,1.263111,0.000000,0.514684,1.263111,26.263111,0.0,2.967732,1.444471
1,None,NaN,0.000000,0.0,0.000000,-0.000000,NaN,NaN,0.000000,0.0,...,0.511675,11.133702,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.512445
2,None,NaN,0.000000,0.0,0.000000,-0.000000,NaN,NaN,0.000000,0.0,...,0.520546,11.013111,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.575787
3,None,NaN,0.000000,0.0,0.000000,-0.000000,NaN,NaN,0.000000,0.0,...,0.527010,10.909558,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.630179
4,None,NaN,0.000000,0.0,0.000000,-0.000000,NaN,NaN,0.000000,0.0,...,0.531626,10.830099,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.671916
5,None,NaN,0.000000,0.0,0.000000,-0.000000,NaN,NaN,0.000000,0.0,...,0.531626,10.482963,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.698153
6,sq,1.0,10.000000,0.0,0.706667,10.000000,NaN,NaN,0.000000,0.0,...,0.430819,3.000000,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.707102
7,sq,1.0,10.000000,0.0,0.913333,10.000000,NaN,NaN,0.000000,0.0,...,0.430819,9.835915,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.698153
8,sq,1.0,4.193548,0.0,1.000000,4.193548,3.0,1.0,10.000000,0.0,...,0.422330,8.551438,0.000000,0.000000,0.514684,0.000000,26.263111,0.0,2.967732,1.671916
9,sq,1.0,0.000000,0.0,1.000000,0.000000,3.0,1.0,10.000000,0.0,...,0.422330,8.585786,0.000000,6.615568,0.425765,-6.615568,26.263111,0.0,2.967732,1.630179


In [58]:
try:
    base_dir = Path(__file__).parent  # works when running as a script
except NameError:
    base_dir = Path.cwd()  # fallback for Jupyter or interactive mode

charging_point_file_path = base_dir / "Data" / "CP_sessions_2023.xlsx"
charging_point_data = pd.read_excel(charging_point_file_path, sheet_name=None)
building_file_path = base_dir / "Data" / "Building_test_data.xlsx"
building_data = pd.read_excel(building_file_path, index_col = [0], sheet_name=None)

prices_file_path = base_dir / "Data" / "prices_2023.xlsx"
prices = pd.read_excel(prices_file_path)
prices.index = pd.DatetimeIndex(prices.iloc[:,0])

In [ ]:
start_date = pd.to_datetime("2023-01-01 00:00:00")
subscription_fee = 605/30
transmission_fee = 0.113
transmission_health_incentive = 0.04
peak_effect_fee = 61.55/30
energy_tax = 0.439
energy_certificate = 0.005
incentive_per_kwh=0.1
days = 5
horizon_hours = 36
store_hours = 24
previous_monthly_peak = 0
current_month=1
previous_soc = pd.DataFrame(index=range(1), columns=list(building_data.keys()))
previous_soc.iloc[:, :] = 0.5
building_on = 1
unfeasible_days = []
# Store rolling results
rolling_results = pd.DataFrame()

# Track ongoing EV sessions across days
ongoing_sessions = {cp_name: [] for cp_name in charging_point_data.keys()}

for day in range(days):
    print("=" * 40)
    print(f"🔄 Day {day + 1} Optimization ({horizon_hours}h Horizon)")

    resolution = int(3600/(prices['Spot prices'].index[1] - prices['Spot prices'].index[0]).total_seconds())

    if resolution == 1:
        freq_index = '60min'
    elif resolution == 4:
        freq_index = '15min'
    else:
        print('Resolution of spot prices is not 15 min or 60 min!! - CHECK')
        break
    
    start_time = start_date + timedelta(days=day)
    end_time = start_time + timedelta(hours=horizon_hours) - pd.Timedelta(seconds=1)
    opt_end_time = start_time + timedelta(hours=store_hours) - pd.Timedelta(seconds=1)
    full_index = pd.date_range(start=start_time,end=end_time,freq=f'{freq_index}')
    
    # 1. Build buildings
    print("Step 1: Initializing buildings")
    building_list = []
    if building_on == 1:
        for name in building_data.keys():
            
            load = building_data[name].loc[start_time:end_time - timedelta(hours=1), 'electricity_load']
            load = load[~load.index.duplicated(keep='first')]
            load = load.reindex(full_index, method = 'ffill') / resolution

            pv = building_data[name].loc[start_time:end_time - timedelta(hours=1), 'pv_production']
            pv = pv[~pv.index.duplicated(keep='first')]
            pv = pv.reindex(full_index, method = 'ffill') / resolution

            bess_capacity = building_data[name]['bess_capacity'].iloc[0]
            bess_max_power = building_data[name]['bess_power'].iloc[0]

            print(f"  - Building {name} with initial SOC {previous_soc[name].iloc[0]}")
            building_list.append(building(
                name=name,
                load=load,
                pv_production=pv,
                bess_capacity=bess_capacity,
                bess_initial_soc=previous_soc[name].iloc[0],
                bess_max_power=bess_max_power
            ))
    else:
        print('Skipped buildings!!!')
    
    # 2. Charging Points
    print("Step 2: Preparing charging points")
    charging_point_list = []
    new_ongoing_sessions = {cp_name: [] for cp_name in charging_point_data.keys()}
    new_sessions_starting_tomorrow = []

    for cp_name in charging_point_data.keys():
        print(f"  ⛽ Charging Point: {cp_name}")
        cp_df = charging_point_data[cp_name]
        capacities, max_powers, arrivals, departures = [], [], [], []
        arrival_socs, desired_socs, ev_ids, session_ids = [], [], [], []

        # a) Add ongoing sessions
        print("   ↪ Checking ongoing sessions from previous day")
        for idx, session in enumerate(ongoing_sessions[cp_name]):
            if session['departure_time'] > start_time:
                dep_time = int((session['departure_time'] - start_time).total_seconds() // (60 * 60 / resolution))
                ev_index = len(capacities)
                session['ev_index'] = ev_index
                capacities.append(session['capacity'])
                max_powers.append(session['max_power'])
                arrivals.append(0)
                departures.append(min(dep_time, horizon_hours * 4))
                arrival_socs.append(session['last_soc'])
                if session['departure_time'] < end_time:
                    desired_socs.append(session['last_desired_soc'])
                else:
                    desired_socs.append(session['last_desired_soc'])
                ev_ids.append(session['ev_id'])
                session_ids.append(session['session_id'])

                print(f"     ✅ Continued EV{session['ev_id']}: dep_time_slot={dep_time}, SOC={session['last_soc']}")
                if session['departure_time'] > opt_end_time:
                    new_ongoing_sessions[cp_name].append(session)

        # b) Add new sessions that start in first 24 hours
        print("   ↪ Adding new sessions starting in first 24 hours")
        today_sessions = cp_df[
            (cp_df['Arrival'] >= start_time) &
            (cp_df['Arrival'] < start_time + timedelta(hours=store_hours))
        ]

        for idx, row in today_sessions.iterrows():
            arrival_time = int((row['Arrival'] - start_time).total_seconds() // (60 * 60 / resolution))
            departure_time = int((row['Departure'] - start_time).total_seconds() // (60 * 60 / resolution))

            capacities.append(row['Capacity'])
            max_powers.append(row['Max_Power'])
            arrivals.append(arrival_time)

            if row['Departure'] > end_time:
                connected_time = departure_time - arrival_time
                slope_linear = (row['Desired SOC'] - row['Arrival SOC']) / connected_time
                linear_requested = slope_linear * (horizon_hours * resolution - arrival_time)
                departures.append(horizon_hours * resolution) #need to fix if they connect just before the end of horizon then what should be desired soc?
                session = {
                    'capacity': row['Capacity'],
                    'max_power': row['Max_Power'],
                    'departure_time': row['Departure'],
                    'last_soc': None,
                    'desired_soc': row['Arrival SOC'] + linear_requested, #row['Desired SOC'],
                    'last_desired_soc': row['Desired SOC'],
                    'cp_name': cp_name,
                    'ev_index': len(capacities) - 1,
                    'ev_id': row['ev_id'],
                    'session_id': row['session_id']
                }
                new_ongoing_sessions[cp_name].append(session)
                new_sessions_starting_tomorrow.append(session)
                print(f"     ➕ New EV (spans days): arrival {arrival_time}, will depart next day and can end day with SOC: {row['Arrival SOC'] + linear_requested}")
                
            elif row['Departure'] >= opt_end_time:
                departures.append(departure_time)
                session = {
                    'capacity': row['Capacity'],
                    'max_power': row['Max_Power'],
                    'departure_time': row['Departure'],
                    'last_soc': None,
                    'desired_soc': row['Desired SOC'],
                    'last_desired_soc': row['Desired SOC'],
                    'cp_name': cp_name,
                    'ev_index': len(capacities) - 1,
                    'ev_id': row['ev_id'],
                    'session_id': row['session_id']
                }
                print(f"     ➕ New EV (spans days): arrival {arrival_time}, will depart next day at: {departure_time}")
                new_ongoing_sessions[cp_name].append(session)
                new_sessions_starting_tomorrow.append(session)
            else:
                departures.append(departure_time)
                print(f"     ➕ New EV: arrival {arrival_time}, departure {departure_time}")

            arrival_socs.append(row['Arrival SOC'])
            desired_socs.append(row['Desired SOC'])
            ev_ids.append(row['ev_id'])
            session_ids.append(row['session_id'])

        charging_point_list.append(charging_point(
            name=cp_name,
            ev_id = ev_ids,
            session_id= session_ids,
            ev_capacity=capacities,
            ev_max_power=max_powers,
            ev_arrival=arrivals,
            ev_departure=departures,
            ev_arrival_soc=arrival_socs,
            ev_desired_soc=desired_socs
        ))

    # 3. Run optimization
    print("Step 3: Solving optimization model")
    spot_prices = np.array(prices['Spot prices'].loc[start_time:end_time] * 11.1 / 1000) #€/MWh to SEK/kWh
    
    opt_model = LEC_Opt_spot(charging_point_list, building_list, spot_prices, resolution=resolution ,v2g_on=1, dc = False, subscription_fee = subscription_fee, transmission_fee = transmission_fee,
                 transmission_health_incentive = transmission_health_incentive, peak_effect_fee = peak_effect_fee, energy_tax = energy_tax, energy_certificate = energy_certificate, incentive_per_kwh=incentive_per_kwh)
    results_df = opt_model.solve()
    if (results_df.solver.termination_condition == TerminationCondition.infeasible) or (results_df.solver.termination_condition == TerminationCondition.infeasibleOrUnbounded):
        print('Model is infeasible!!!!')
        break
    elif (results_df.solver.termination_condition == TerminationCondition.infeasible):
        print('Model is unbounded!! check the decision variable bounds!!')
        break
    elif (results_df.solver.termination_condition == TerminationCondition.optimal):
        print('Optimial solution found')
        df = opt_model.get_results()
        df.index = prices.loc[start_time:end_time, 'Spot prices'].index
    else:
        print(f'Solver stopped with condition: {results_df.solver.termination_condition}')
        break
    print("   ✅ Optimization complete")
    for col in df.columns:
        if '_soc' in col:
            print(col, "→", df[col].iloc[store_hours * resolution - 1])
    print("\n✅ Columns in results DataFrame:")
    print([col for col in df.columns if '_soc' in col])

    # Save first 24h of results
    print("Step 3: Saving 24h results to cumulative DataFrame")
    rolling_results = pd.concat([rolling_results, df.iloc[:store_hours * resolution]])
    opt_month = rolling_results.index[-1].month
    if current_month - opt_month == 0:
        opt_peak = (df['P_import'].iloc[:store_hours * resolution] - df['P_export'].iloc[:store_hours * resolution]).max()   #needs to be from the opt
        if previous_monthly_peak < opt_peak:
            previous_monthly_peak = opt_peak
    else:
        previous_monthly_peak = 0
        current_month = opt_month

    # 4. Update building SOCs
    if building_on == 1:
        print("Step 4: Updating building SOCs")
        for name in building_data.keys():
            soc_val = df[f'{name}_bess_soc'].iloc[store_hours * resolution - 1]
            previous_soc[name].iloc[0] = soc_val
            print(f"  🔋 {name} end-of-day SOC: {soc_val:.2f}")
    else:
        print('Step 4. No buildings available')

    # 5. Update ongoing sessions' SOCs
    print("Step 5: Updating ongoing session SOCs from Day", day + 1)
    for cp_name in ongoing_sessions.keys():
        for session in ongoing_sessions[cp_name]:
            ev_name = f"{session['cp_name']}"
            if ev_name + "_soc" in df.columns:
                session['last_soc'] = df[f'{ev_name}_soc'].iloc[store_hours * resolution - 1]
                print(f"🔄 Updated {ev_name} SOC = {session['last_soc']:.2f}")
            else:
                print(f"⚠️ Warning: {ev_name}_soc not found in results")

    # 6. Update new sessions for tomorrow with today’s end SOC
    print("Step 6: Updating sessions starting today that continue to tomorrow")
    for session in new_sessions_starting_tomorrow:
        ev_name = f"{session['cp_name']}"
        if f"{ev_name}_soc" in df.columns:
            session['last_soc'] = float(df[f"{ev_name}_soc"].iloc[store_hours * resolution - 1])
            print(f"🚚 {ev_name}: SOC carried to next day = {session['last_soc']:.2f}")
        else:
            print(f"⚠️ Could not update SOC for {ev_name}")

    # 7. Carry forward ongoing sessions
    print("Step 7: Updating ongoing_sessions for next day")
    ongoing_sessions = new_ongoing_sessions

    print("=" * 40 + "\n")
rolling_results['Cyc_Age'] = rolling_results.filter(regex='_CycAg$').sum(axis=1) * 100
rolling_results['Cal_Age'] = rolling_results.filter(regex='_CalAg$').sum(axis=1) * 100
rolling_results['Cyc_Cost'] = rolling_results.filter(regex='_CycCost$').sum(axis=1) 
rolling_results['Cal_Cost'] = rolling_results.filter(regex='_CalCost$').sum(axis=1) 
peak_load = (rolling_results['P_import']-rolling_results['P_export']).resample('M').max()
monthly_peak_costs = peak_load * peak_effect_fee / 24 / resolution
month_end_index = rolling_results.index.to_period('M').to_timestamp('M')
rolling_results['Peak cost'] = month_end_index.map(monthly_peak_costs)
rolling_results['DSO cost'] = rolling_results['Transmission cost'] + rolling_results['Peak cost']
rolling_results['Tax cost'] = 0.25 * (rolling_results['Supplier cost'] + rolling_results['DSO cost']) + 1.25 * energy_tax * (rolling_results['P_import'] - rolling_results['P_export']) / resolution
rolling_results['Overall cost'] = rolling_results['DSO cost'] + rolling_results['Supplier cost'] + rolling_results['Tax cost']
print(f'Overall simulation completed - number of unfesible days: {len(unfeasible_days)} and they are: {unfeasible_days}')

🔄 Day 1 Optimization (36h Horizon)
Step 1: Initializing buildings
  - Building B1 with initial SOC 0.5
  - Building B2 with initial SOC 0.5
Step 2: Preparing charging points
  ⛽ Charging Point: CP1
   ↪ Checking ongoing sessions from previous day
   ↪ Adding new sessions starting in first 24 hours
     ➕ New EV (spans days): arrival 14, will depart next day and can end day with SOC: 0.5958556731093458
  ⛽ Charging Point: CP2
   ↪ Checking ongoing sessions from previous day
   ↪ Adding new sessions starting in first 24 hours
     ➕ New EV: arrival 39, departure 80
Step 3: Solving optimization model
Optimial solution found
Objective value: 190.91307160701854
⏱ Time steps in model: 144
   ✅ Optimization complete
CP1_soc → 0.2
CP2_soc → 0.0
B1_bess_soc → 0.2
B2_bess_soc → 0.2

✅ Columns in results DataFrame:
['CP1_soc', 'CP2_soc', 'B1_bess_soc', 'B2_bess_soc']
Step 3: Saving 24h results to cumulative DataFrame
Step 4: Updating building SOCs
  🔋 B1 end-of-day SOC: 0.20
  🔋 B2 end-of-day SOC

In [54]:
rolling_results['Overall cost'].sum()

np.float64(524.881612545503)

In [47]:
rolling_results

,CP1_EV_connection,CP1_Session,CP1_ch,CP1_ds,CP1_soc,CP1_P,CP2_EV_connection,CP2_Session,CP2_ch,CP2_ds,...,Transmission cost,Supplier cost,Cyc_Age,Cal_Age,Cyc_Cost,Cal_Cost,Peak cost,DSO cost,Tax cost,Overall cost
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,NaN,NaN,0.0,0.0,0.00000,-0.0,NaN,NaN,0.000000,0.0,...,0.144813,0.006701,0.0,0.0,0.0,0.0,0.157811,0.302623,0.780569,1.089893
2023-01-01 00:15:00,NaN,NaN,0.0,0.0,0.00000,-0.0,NaN,NaN,0.000000,0.0,...,0.122865,0.005685,0.0,0.0,0.0,0.0,0.157811,0.280676,0.668247,0.954608
2023-01-01 00:30:00,NaN,NaN,0.0,0.0,0.00000,-0.0,NaN,NaN,0.000000,0.0,...,0.144813,0.006701,0.0,0.0,0.0,0.0,0.157811,0.302623,0.780569,1.089893
2023-01-01 00:45:00,NaN,NaN,0.0,0.0,0.00000,-0.0,NaN,NaN,0.000000,0.0,...,0.071367,0.003302,0.0,0.0,0.0,0.0,0.157811,0.229177,0.404690,0.637169
2023-01-01 01:00:00,NaN,NaN,0.0,0.0,0.00000,-0.0,NaN,NaN,0.000000,0.0,...,0.144813,0.006609,0.0,0.0,0.0,0.0,0.157811,0.302623,0.780546,1.089778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-01-05 22:45:00,12.0,5.0,0.0,0.0,0.64739,0.0,57.0,5.0,1.837522,0.0,...,0.178470,0.033264,0.0,0.0,0.0,0.0,0.157811,0.336281,0.959071,1.328616
2023-01-05 23:00:00,12.0,5.0,0.0,0.0,0.64739,0.0,57.0,5.0,2.305022,0.0,...,0.178470,0.030718,0.0,0.0,0.0,0.0,0.157811,0.336281,0.958435,1.325433
2023-01-05 23:15:00,12.0,5.0,0.0,0.0,0.64739,0.0,57.0,5.0,2.305022,0.0,...,0.178470,0.030718,0.0,0.0,0.0,0.0,0.157811,0.336281,0.958435,1.325433
